# Bayes' Theorem: The Spam Example

This notebook applies Bayes' theorem to a real machine-learning task — **spam classification** — and introduces the vocabulary of Bayesian inference: the **prior** (what you believe before seeing evidence) and how a feature updates it. It is based on the twelfth video of Week 1 (Introduction to Probability).


## 1. Introduction

You have a dataset of **100 emails**, and **20 of them are spam**.

The simplest possible classifier says: *every email is spam with probability 20%* — that's all we know so far. To do better, we look for a **feature**: spam emails tend to contain words like **"lottery"**. Counting the dataset:

- of the 20 spam emails, **14** contain "lottery";
- of the 80 non-spam emails (called **ham**), **10** contain "lottery".

Question: what is the probability that an email containing "lottery" is spam — $P(\text{spam} \mid \text{lottery})$?

After completing this notebook you will be able to:

- Compute a Bayesian posterior by filtering a dataset down to the rows matching the condition.
- Identify the **prior** and the likelihoods in a classification problem.
- Verify that the Bayes formula reproduces the filtered count exactly.


## 2. Theory

### Bayes as deletion

This is the key intuition of the video: using Bayes' theorem means **deleting all the emails that don't matter** — the ones without the word "lottery" — and computing a simple probability among the survivors:

$$
P(\text{spam} \mid \text{lottery}) = \frac{\text{spam emails containing lottery}}{\text{all emails containing lottery}}
$$

### The vocabulary of Bayesian inference

- **Prior** — $P(\text{spam}) = 0.2$: the probability before seeing any evidence. The most basic piece of knowledge in the problem.
- **Likelihoods** — $P(\text{lottery} \mid \text{spam})$ and $P(\text{lottery} \mid \text{ham})$: how the evidence behaves in each class.
- **Posterior** — $P(\text{spam} \mid \text{lottery})$: the updated probability after seeing the evidence.

### The formula

With $A$ = spam and $B$ = contains "lottery":

$$
P(A \mid B) = \frac{P(A)\,P(B \mid A)}{P(A)\,P(B \mid A) + P(A')\,P(B \mid A')}
$$


## 3. Worked Example

### By filtering

Only the emails containing "lottery" matter: $14 + 10 = 24$ of them. Among these, 14 are spam:

$$
P(\text{spam} \mid \text{lottery}) = \frac{14}{24} = \frac{7}{12} \approx 0.583
$$

### By the formula

Compute each ingredient from the dataset:

| Quantity | Count | Value |
|---|---|---|
| $P(\text{spam})$ — the **prior** | 20 of 100 | 0.2 |
| $P(\text{ham})$ | 80 of 100 | 0.8 |
| $P(\text{lottery} \mid \text{spam})$ | 14 of 20 | 0.7 |
| $P(\text{lottery} \mid \text{ham})$ | 10 of 80 | 0.125 |

Plug in:

$$
P(\text{spam} \mid \text{lottery}) = \frac{0.2 \times 0.7}{0.2 \times 0.7 + 0.8 \times 0.125}
= \frac{0.14}{0.14 + 0.10} = \frac{0.14}{0.24} \approx 0.583
$$

Same answer — the formula *is* the filtering, done with proportions instead of raw counts. Seeing "lottery" moved the spam probability from the prior 20% to over 58%.


## 4. Python Examples


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from fractions import Fraction

rng = np.random.default_rng(seed=42)


### Building the dataset and filtering

We recreate the 100 emails as a small table and compute the posterior the way the video does: keep only the rows with the condition.


In [ ]:
# Columns: is_spam, has_lottery — recreated from the video's counts
emails = ([(True, True)] * 14 + [(True, False)] * 6 +      # 20 spam
          [(False, True)] * 10 + [(False, False)] * 70)     # 80 ham
emails = np.array(emails)
is_spam, has_lottery = emails[:, 0], emails[:, 1]

with_lottery = emails[has_lottery]
p_filtered = with_lottery[:, 0].mean()

print(f"Emails containing 'lottery': {len(with_lottery)}")
print(f"...of which spam:            {with_lottery[:, 0].sum()}")
print(f"P(spam | lottery) = 14/24 = {Fraction(14, 24)} = {p_filtered:.3f}")


### The formula, ingredient by ingredient


In [ ]:
p_spam = is_spam.mean()                            # the prior
p_lottery_given_spam = has_lottery[is_spam].mean()
p_lottery_given_ham = has_lottery[~is_spam].mean()

print(f"Prior           P(spam)          = {p_spam}")
print(f"Likelihood      P(lottery | spam) = {p_lottery_given_spam}")
print(f"Likelihood      P(lottery | ham)  = {p_lottery_given_ham}")

numerator = p_spam * p_lottery_given_spam
posterior = numerator / (numerator + (1 - p_spam) * p_lottery_given_ham)
print(f"Posterior       P(spam | lottery) = {posterior:.3f}")


### From probability to classifier

A Bayesian spam filter classifies an email as spam if the posterior crosses a threshold (say 0.5). With one feature, our tiny classifier already beats the always-20% baseline on lottery emails.


In [ ]:
def posterior_spam(p_spam, p_feat_spam, p_feat_ham, has_feature):
    if not has_feature:      # use complements for emails WITHOUT the word
        p_feat_spam, p_feat_ham = 1 - p_feat_spam, 1 - p_feat_ham
    num = p_spam * p_feat_spam
    return num / (num + (1 - p_spam) * p_feat_ham)

for has_word in [True, False]:
    p = posterior_spam(0.2, 0.7, 0.125, has_word)
    label = "spam" if p > 0.5 else "ham"
    word = "contains" if has_word else "doesn't contain"
    print(f"Email {word:>16} 'lottery': P(spam) = {p:.3f} -> classify as {label}")


## 5. Visualizations

### Bayes as deletion

Each square is one email. Bayes' theorem grays out everything without the word "lottery" — among the 24 that remain, 14 are spam.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))

# Order emails: spam+lottery, spam only, ham+lottery, ham only
groups = [(14, "#e15759", 1.0), (6, "#e15759", 0.18),
          (10, "#4e79a7", 1.0), (70, "#4e79a7", 0.18)]
i = 0
for count, color, alpha in groups:
    for _ in range(count):
        x, y = i % 25, 3 - i // 25
        ax.add_patch(plt.Rectangle((x, y), 0.88, 0.88,
                                   facecolor=color, alpha=alpha))
        i += 1

ax.set_xlim(-0.2, 25)
ax.set_ylim(-1.6, 4.1)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("100 emails — spam red, ham blue; faded = no 'lottery' (deleted by the condition)")
ax.text(12.5, -1.1, "Kept: 14 spam + 10 ham = 24 emails → P(spam | lottery) = 14/24 ≈ 0.583",
        ha="center", fontsize=11,
        bbox=dict(boxstyle="round", facecolor="#fff3cd"))
plt.show()


### Prior vs. posterior

One word of evidence nearly triples the spam probability.


In [ ]:
labels = ["Prior\nP(spam)", "Posterior\nP(spam | lottery)"]
values = [0.2, 14 / 24]

plt.figure(figsize=(7, 5))
bars = plt.bar(labels, values, color=["#9ecae1", "#e15759"])
for bar, v in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width() / 2, v + 0.015, f"{v:.3f}",
             ha="center", fontsize=12)
plt.ylim(0, 0.75)
plt.title("Evidence updates belief: seeing 'lottery' moves 0.2 to 0.583")
plt.ylabel("Probability of spam")
plt.grid(True, alpha=0.3, axis="y")
plt.show()


### The whole story on one page

Everything above in a single sheet: all 100 emails, the 24 that survive the deletion, the prior-to-posterior update, and the one-word classifier.

<img src="img/12_bayes_spam_deletion.png" alt="Concept sheet for this notebook" width="900">

## 6. Exercises

### Basic

1. What is $P(\text{ham} \mid \text{lottery})$?
2. In the same dataset, what is $P(\text{spam} \mid \text{no lottery})$? Use the counts: 6 spam and 70 ham emails lack the word.

### Intermediate

3. A second feature: the word "winner" appears in 12 of the 20 spam emails and 4 of the 80 ham emails. Compute $P(\text{spam} \mid \text{winner})$.
4. Suppose the dataset had 50 spam and 50 ham emails, with the same likelihoods (70% and 12.5%). Recompute $P(\text{spam} \mid \text{lottery})$ — how much does the prior matter?

### Advanced

5. Assume "lottery" and "winner" appear **independently within each class** (the naive Bayes assumption). Compute $P(\text{spam} \mid \text{both words})$ by using the posterior after "lottery" as the prior for the "winner" update, and confirm you get the same result in one shot with the product of likelihoods.


## 7. Solutions


In [ ]:
def bayes(p_a, p_b_given_a, p_b_given_not_a):
    num = p_a * p_b_given_a
    return num / (num + (1 - p_a) * p_b_given_not_a)

# Exercise 1
print("1.  P(ham | lottery) = 10/24 =", round(10 / 24, 3))

# Exercise 2
print("2.  P(spam | no lottery) = 6/76 =", round(6 / 76, 3),
      "- absence of the word is evidence of ham")

# Exercise 3
print("3.  P(spam | winner) =", round(bayes(0.2, 12 / 20, 4 / 80), 3))

# Exercise 4: balanced prior
print("4.  With P(spam) = 0.5:", round(bayes(0.5, 0.7, 0.125), 3),
      "- the same evidence is far more damning with a higher prior")

# Exercise 5: two-word update, sequentially and in one shot
after_lottery = bayes(0.2, 0.7, 0.125)
after_both_seq = bayes(after_lottery, 12 / 20, 4 / 80)

num = 0.2 * 0.7 * (12 / 20)
den = num + 0.8 * 0.125 * (4 / 80)
after_both_once = num / den
print(f"5.  Sequential: {after_both_seq:.4f}, one-shot: {after_both_once:.4f} "
      "- identical (naive Bayes)")


## 8. Conclusion

- Bayes' theorem in practice = **filter to the condition, then count**: 24 emails contain "lottery", 14 are spam, posterior $\frac{7}{12} \approx 0.583$.
- The **prior** ($P(\text{spam}) = 0.2$) is your belief before evidence; the feature updates it to the **posterior** (0.583).
- The formula and the filtering always agree — the formula just works with proportions instead of raw counts.
- Stacking independent features by chaining updates is exactly the **naive Bayes classifier**, one of the simplest and most enduring ML algorithms.
